In [ ]:
'''libraries'''
#Data
import pandas as pd

#Plots
import matplotlib.pyplot as plt

#fits
#from scipy.optimize import curve_fit

#math
import numpy as np
import scipy.integrate as integrate


#Constants
from scipy.constants import physical_constants
m_u=physical_constants['atomic mass constant energy equivalent in MeV'][0]
from scipy.constants import speed_of_light as c

#Usefull
from tqdm.notebook import tqdm
import os
from scipy.interpolate import interp1d
import pynucastro as pyna
#%matplotlib widget

In [ ]:
initial_time=1 #seconds
final_time=4.32E7 #seconds 500 days= 4.32E7 seconds 100 days= 8.64E7 seconds
N_steps=100000
time = np.exp(np.linspace(np.log(initial_time), np.log(final_time), N_steps))

def dudt_funtion_of_E_t(ta,u,b,tc,E__):
    if ta<=1:
        return (E__[np.searchsorted(time,ta*tc)])/(ta**3)-u*((4/ta) + 3*ta/(4*b))
    elif ta>1:
        return (E__[np.searchsorted(time,ta*tc)])/(ta**3)-(u/ta)*(4+3/(4*b))
 
def solucionar(U_0,tau_n,b,tc,n,E__):
    U_n=np.zeros(n)
    U_n[0]=U_0
    
    h=tau_n[1]-tau_n[0]
    for i in range(len(U_n)):
    
        k1=h*dudt_funtion_of_E_t(tau_n[i],U_n[i],b,tc,E__)
        k2=h*dudt_funtion_of_E_t(tau_n[i]+h/2,U_n[i]+k1/2,b,tc,E__)
        k3=h*dudt_funtion_of_E_t(tau_n[i]+h/2,U_n[i]+k2/2,b,tc,E__)
        k4=h*dudt_funtion_of_E_t(tau_n[i]+h,U_n[i]+k3,b,tc,E__)
        if i!=len(U_n)-1:
            U_n[i+1]=U_n[i]+(1/6)*(k1+2*k2+2*k3+k4)
    return U_n

def generar_graficas(M,k,b,U_0,tau_0,tau_f,n,E_):
    M=M*(1.989*10**30)#kg
    tc=np.sqrt(3*k*M/(4*np.pi*(b*c)**2))
    
    Tau_n=np.linspace(tau_0,tau_f,n)
    Uo=solucionar(U_0,Tau_n,b,tc,n,E_)
    te=tc*Tau_n
    tedias=te*(1/(60*60*24))
    U=((3*M)/(4*np.pi*((b*c)**3)*((tc)**(2))))*Uo
    L=np.zeros(len(U))
    i_tc=np.searchsorted(te,tc)
    L[:i_tc]=((np.pi*((b*c)**2)*c)*(te[:i_tc]**4)/(tc**2))*U[:i_tc]*10**7
    calpha=L[i_tc-1]/E_[np.searchsorted(time,tc)]
    #print(calpha,L[i_tc-1],E_[np.searchsorted(t,tc)])    
    #L[i_tc:]=((np.pi*c*(b*c)**2))*(te[i_tc:]**2)*U[i_tc:]*10**7
    
    spline_interp = interp1d(time, E_, kind='linear', fill_value="extrapolate")
    e_new=spline_interp(te)
    L[i_tc:]=calpha*e_new[i_tc:]
    T=(calpha,L[i_tc],E_[np.searchsorted(time,tc)])  
    #T=0#(c*U/(Sb*4))**(1/4)
    
    
    return [U,L,T,tedias]